# Pattern 2: Tool Use

Tool use was discussed in a [previous notebook](/topics/agents/01.html#function-calling) using the OpenAI client. There we considered an LLM as a [brain in a vat](@fig-brain-vat) that hallucinates tool call depending on the list of tools provided to it and the task at hand. We also discussed the [Toolformer architecture](/topics/agents/#toolformer-toolformer) [@toolformer] which trained a base GPT-J-6b for tool calling. Moreover, we saw that tool calling *emerges* at around 755M parameters for GPT-2. 

In this notebook, we discuss tool calling in practice that is a bit more API agnostic (i.e. we don't just use the `tools` API of the LLM client). We also define a `@tool` decorator which allows us to automatically convert a function into a tool schema that we can inject in the system prompt as text.

![**LLM as brain in a vat.** The LLM as core reasoning module thinking of what tools to call without having the capability to execute them. We provide a separate process and environment for running the code. [Source](https://en.wikipedia.org/wiki/Brain_in_a_vat) ](./img/brain-vat.png){#fig-brainvat width=70%}

## Setting up

In [6]:
from notebooks.utils import load_dotenv
from notebooks.agents.chat import ChatHistory, completions_create
from notebooks.agents.utils import extract_tag_content, get_client


load_dotenv(verbose=True)
client = get_client("groq")()
MODEL = "llama-3.3-70b-versatile"

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


## System prompt

**Example tool.** For the example below, we use the [Weather Forecast API](https://open-meteo.com/en/docs):

In [7]:
import json
import requests

def get_weather(latitude, longitude):
    """
    Get current weather data for provided coordinates. Returns weather data
    with units: temperature (celsius), wind speed (kph), & precipitation (mm).
    """
    response = requests.get((
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&"
        "current=temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation,precipitation_probability"
    ))
    data = response.json()
    return data["current"]


get_weather(latitude=14.4779, longitude=121.3214)  # true coordinates

{'time': '2025-09-09T17:45',
 'interval': 900,
 'temperature_2m': 28.1,
 'wind_speed_10m': 5.6,
 'relative_humidity_2m': 82,
 'precipitation': 0.0,
 'precipitation_probability': 0}

**Defining the system prompt.** Note the 6 strict rules provided to ensure that the LLM follows the function schema. We also provide few-shot examples to help the LLM better understand our schema. Note that the models [[openai/gpt-oss-120b]](https://huggingface.co/openai/gpt-oss-120b) and [[llama-3.3-70b]](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_3/) are capable of tool calling but were trained with different schemas.

In [8]:
tool_system_prompt_template = lambda tools: f"""
You are an AI assistant designed to call external functions. Your primary role is to analyze the user's request and execute the appropriate function calls based on the tools provided.

## STRICT RULES:
1.  **TOOL SELECTION:** You MUST only call functions defined in the <tools> section. Calling an undefined function is a critical error.
2.  **ARGUMENT STRICTNESS:** You MUST provide all `required_arguments` and MAY provide any `optional_arguments`. You MUST NOT provide any arguments not defined in the function's schema.
3.  **DATA TYPES:** You MUST respect the `type` of each argument (e.g., `string`, `number`, `boolean`, `array`, `object`).
4.  **INFERENCE:** You MUST infer argument values from the user's query. If a user mentions a location like "Paris," you must provide its coordinates for a function that requires `latitude` and `longitude`.
5.  **MULTIPLE CALLS:** You MAY call one or more functions in sequence to fully satisfy the user's request.
6.  **OUTPUT FORMAT:** You MUST output each function call in the exact JSON format specified, wrapped in <tool_call></tool_call> tags.

## OUTPUT INSTRUCTIONS:
For each function you decide to call, output a JSON object with the following structure inside <tool_call></tool_call> tags:
{{"name": "function_name", "arguments": {{"arg1": "value1", "arg2": 123}}, "id": <monotonically_increasing_integer>}}
-   `id`: Start from 1 and increment by 1 for each subsequent call in your response.

## AVAILABLE TOOLS:
The following functions are available for you to call. Study their names, descriptions, and argument schemas carefully.

<tools>
{tools}
</tools>

## DECISION PROCESS:
1.  Identify the user's intent.
2.  Find the most relevant tool(s) to fulfill that intent.
3.  For each tool, extract or infer all required arguments from the user's query. If an optional argument can be inferred, include it.
4.  If a required argument cannot be inferred, you MUST ask the user for clarification. Do not guess.
5.  Output the function call(s) in the specified format.

## EXAMPLES:

Example 1: Single Function Call
User: "What's the weather like in Tokyo?"
Output:
<tool_call>
{{"name": "get_weather", "arguments": {{"latitude": 35.6762, "longitude": 139.6503}}, "id": 1}}
</tool_call>

Example 2: Function with Optional Arguments
User: "Find me some cheap Italian food in Manila."
Output:
<tool_call>
{{"name": "search_restaurants", "arguments": {{"location": "Manila", "cuisine": "Italian", "max_price": 1}}, "id": 1}}
</tool_call>

Example 3: Missing Required Argument
User: "Send an email to john@example.com."
Output:
I cannot send the email yet. I need to know the subject and body of the message. What would you like the email to say?
"""

Note that the tool schema is different from the [OpenAI spec](https://platform.openai.com/docs/guides/function-calling#defining-functions):

In [9]:
get_fn = {
    "get_weather": get_weather
}

tools = """[
    {
        "name": "get_weather",
        "description": "Get current weather data for provided coordinates with units: temperature (celsius), wind speed (kph), & precipitation (mm).",
        "arguments": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required_arguments": ["latitude", "longitude"],
        "optional_arguments": null
    }
]"""

TOOL_SYSTEM_PROMPT = tool_system_prompt_template(tools)

Example user prompt:

In [10]:
chat = ChatHistory(TOOL_SYSTEM_PROMPT)
chat.update(role="user", prompt="What's the weather like in Quisao, Pililla, Rizal right now?")
response = completions_create(client, chat, model=MODEL)

# parsing the response
response_extract = extract_tag_content(response, tag="tool_call")
if response_extract.found:
    for call in response_extract.content:
        call_dict = json.loads(call)
        print(call_dict)
        print(get_fn[call_dict["name"]](**call_dict["arguments"]))

{'name': 'get_weather', 'arguments': {'latitude': 14.4833, 'longitude': 121.2667}, 'id': 1}
{'time': '2025-09-09T17:45', 'interval': 900, 'temperature_2m': 27.2, 'wind_speed_10m': 4.0, 'relative_humidity_2m': 90, 'precipitation': 0.0, 'precipitation_probability': 5}
